# 📡 DSP Interactive Notebook
## Digital Signal Processing — Hands-On Exploration

This notebook walks through all major DSP concepts interactively.
Run each cell in order.

**Modules covered:**
1. Signals & Systems
2. Filters (FIR / IIR / Adaptive)
3. Modulation (AM / FM / BPSK / QAM / OFDM)
4. Noise Cancellation
5. Full TX→Channel→RX Chain
6. MIMO & OFDM Systems


In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp

%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('✅ Ready!')

## 1. Signals & Systems

In [ ]:
from signals.generator import sine_wave, square_wave, chirp, awgn, compute_fft

fs = 4000

# Multi-tone signal
t, s1 = sine_wave(100, 1.0, duration=0.05, fs=fs)
t, s2 = sine_wave(500, 0.5, duration=0.05, fs=fs)
t, s3 = sine_wave(1200, 0.3, duration=0.05, fs=fs)
composite = s1 + s2 + s3
noisy, _  = awgn(composite, snr_db=12)

fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].plot(t*1000, composite, color='steelblue', lw=1.5, label='clean')
axes[0].plot(t*1000, noisy,    color='tomato', lw=0.8, alpha=0.7, label='noisy (SNR=12dB)')
axes[0].set_xlabel('ms'); axes[0].set_title('Multi-tone Signal'); axes[0].legend()

freqs, mag = compute_fft(composite, fs)
axes[1].plot(freqs, mag, color='darkorange', lw=2)
axes[1].set_xlabel('Hz'); axes[1].set_title('FFT Spectrum')
plt.suptitle('Signals & Systems', fontweight='bold')
plt.tight_layout()

In [ ]:
# Chirp spectrogram
from signals.generator import plot_spectrogram
t_c, ch = chirp(50, 1500, duration=1.0, fs=fs)
fig = plot_spectrogram(ch, fs, title='Chirp Spectrogram (50 → 1500 Hz)')
plt.show()

## 2. Filters

In [ ]:
from filters.design import (butterworth_lpf, chebyshev1_lpf, elliptic_lpf,
                             fir_window_lpf, apply_iir, apply_fir,
                             plot_frequency_response, notch_filter)

fc = 500  # cutoff Hz

# IIR designs
b_b, a_b = butterworth_lpf(fc, fs, order=5)
b_c, a_c = chebyshev1_lpf(fc, fs, order=5)
b_e, a_e = elliptic_lpf(fc, fs, order=5)

fig = plot_frequency_response({
    'Butterworth N=5' : (b_b, a_b),
    'Chebyshev-I N=5' : (b_c, a_c),
    'Elliptic N=5'    : (b_e, a_e),
}, fs, title='IIR Filter Comparison (cutoff=500 Hz)')
plt.show()

In [ ]:
# Apply filter to noisy signal
t, sig_low  = sine_wave(200, 1.0, duration=0.1, fs=fs)
t, sig_high = sine_wave(1200, 0.8, duration=0.1, fs=fs)
mixed = sig_low + sig_high
clean_out = apply_iir(b_b, a_b, mixed)

fig, axes = plt.subplots(3, 1, figsize=(13, 8))
axes[0].plot(t, mixed,     color='tomato');     axes[0].set_title('Mixed: 200Hz + 1200Hz')
axes[1].plot(t, clean_out, color='steelblue'); axes[1].set_title('After Butterworth LPF (500Hz cutoff)')
axes[2].plot(t, sig_low,   color='seagreen');  axes[2].set_title('Original 200Hz (reference)')
plt.tight_layout()

## 3. Modulation

In [ ]:
from modulation.schemes import (fm_modulate, fm_demodulate,
                                 bpsk_modulate, qam_modulate,
                                 plot_qam_constellation,
                                 ber_bpsk_theory, ber_bpsk_simulation)

fc_mod = 1000

# FM
t_m = np.linspace(0, 0.05, int(fs*0.05))
msg = np.sin(2*np.pi*200*t_m)
s_fm, _ = fm_modulate(msg, fc=fc_mod, fs=fs, kf=50)

fig, axes = plt.subplots(2, 1)
axes[0].plot(t_m*1000, msg,  color='seagreen'); axes[0].set_title('Message (200 Hz)')
axes[1].plot(t_m*1000, s_fm, color='darkorange'); axes[1].set_title('FM Modulated')
plt.suptitle('FM Modulation', fontweight='bold'); plt.tight_layout()

In [ ]:
# 16-QAM constellation
rng  = np.random.default_rng(0)
bits = rng.integers(0, 2, 400)
_, syms = qam_modulate(bits, M=16, fc=fc_mod, fs=fs)
noise   = rng.normal(0, 0.25, len(syms)) + 1j*rng.normal(0, 0.25, len(syms))

fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].scatter(syms.real, syms.imag, s=30, color='steelblue')
axes[0].set_title('16-QAM (clean)'); axes[0].set_aspect('equal'); axes[0].axhline(0,color='k',lw=0.5); axes[0].axvline(0,color='k',lw=0.5)
axes[1].scatter((syms+noise).real, (syms+noise).imag, s=15, alpha=0.5, color='tomato')
axes[1].set_title('16-QAM (with noise)'); axes[1].set_aspect('equal'); axes[1].axhline(0,color='k',lw=0.5); axes[1].axvline(0,color='k',lw=0.5)
plt.suptitle('QAM Constellation', fontweight='bold'); plt.tight_layout()

In [ ]:
# BER vs Eb/N0
snr_range = np.arange(0, 13)
ber_th  = ber_bpsk_theory(snr_range)
ber_sim = ber_bpsk_simulation(snr_range, num_bits=50000)

fig, ax = plt.subplots(figsize=(10,5))
ax.semilogy(snr_range, ber_th,  'b-',  lw=2,  label='Theory')
ax.semilogy(snr_range, ber_sim, 'ro--', lw=1.5, label='Simulation')
ax.set_xlabel('Eb/N₀ (dB)'); ax.set_ylabel('BER')
ax.set_title('BPSK BER vs Eb/N₀'); ax.legend()
ax.set_ylim([1e-5, 1])

## 4. Noise Cancellation

In [ ]:
from noise_cancellation.canceller import (spectral_subtraction, lms_anc, rls_anc, compute_snr)

N   = 4000
t_n = np.arange(N) / fs
desired   = np.sin(2*np.pi*300*t_n)
hum       = np.sin(2*np.pi*50*t_n)
corrupted = desired + 0.8*hum + 0.1*np.random.randn(N)
reference = hum + 0.05*np.random.randn(N)

e_lms, _ = lms_anc(reference, corrupted, mu=0.003, num_taps=64)
e_rls, _ = rls_anc(reference, corrupted, lam=0.995, num_taps=64)

snr_in  = compute_snr(desired, corrupted)
snr_lms = compute_snr(desired, e_lms)
snr_rls = compute_snr(desired, e_rls)

fig, axes = plt.subplots(3, 1, figsize=(13, 9))
axes[0].plot(t_n[:500], corrupted[:500], color='tomato');    axes[0].set_title(f'Corrupted Input (SNR={snr_in:.1f} dB)')
axes[1].plot(t_n[:500], e_lms[:500],    color='steelblue'); axes[1].set_title(f'LMS ANC Output (SNR≈{snr_lms:.1f} dB)')
axes[2].plot(t_n[:500], e_rls[:500],    color='seagreen');  axes[2].set_title(f'RLS ANC Output (SNR≈{snr_rls:.1f} dB)')
plt.suptitle('Active Noise Cancellation', fontweight='bold'); plt.tight_layout()

## 5. OFDM System

In [ ]:
from ofdm.ofdm_system import OFDMConfig, ofdm_tx_frame, ofdm_rx_frame, awgn as ofdm_awgn, compute_papr, papr_ccdf

cfg = OFDMConfig(N=64, cp_len=16, n_pilots=8, mod_order=4)
tx_frame, tx_bits = ofdm_tx_frame(20, cfg)
rx_frame = ofdm_awgn(tx_frame, 20)
rx_bits, rx_syms = ofdm_rx_frame(rx_frame, 20, cfg)

n   = min(len(tx_bits), len(rx_bits))
ber = np.sum(tx_bits[:n] != rx_bits[:n]) / n

fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].plot(tx_frame.real, color='steelblue', lw=0.7); axes[0].set_title('OFDM TX Frame (I)')
axes[1].scatter(rx_syms.real, rx_syms.imag, s=15, alpha=0.5, color='darkorange')
axes[1].set_title(f'QPSK Constellation after Equalisation (BER={ber:.4f})')
axes[1].set_aspect('equal'); axes[1].axhline(0,color='k',lw=0.5); axes[1].axvline(0,color='k',lw=0.5)
plt.suptitle('OFDM System', fontweight='bold'); plt.tight_layout()

print(f'PAPR: {compute_papr(tx_frame):.2f} dB')

## 6. MIMO System

In [ ]:
from mimo.mimo_system import mimo_capacity, plot_mimo_capacity

fig = plot_mimo_capacity(snr_range=np.arange(-5, 31))
plt.show()
print('MIMO capacity plotted for 1×1, 2×2, 4×4, 8×8 configurations')